# Adiabatic Quantum Computation — Concept Demo

Illustrates the adiabatic path: start with a simple Hamiltonian whose ground state is easy to prepare, then slowly evolve to a problem Hamiltonian whose ground state encodes the solution.

The adiabatic theorem guarantees that if the evolution is slow enough relative to the minimum spectral gap, the system stays in the ground state throughout.

In [ ]:
import dimod
import neal
import numpy as np

## Problem and Hamiltonians

In [ ]:
EDGES = [(0, 1), (1, 2), (2, 3), (3, 0)]
N = 4

def cut_size(bits):
    colors = [int(b) for b in bits[::-1]]
    return sum(colors[i] != colors[j] for i, j in EDGES)

def build_problem_bqm():
    bqm = dimod.BinaryQuadraticModel("SPIN")
    for i in range(N):
        bqm.add_variable(i, 0.0)
    for i, j in EDGES:
        bqm.add_interaction(i, j, -1.0)
    return bqm

def build_initial_bqm():
    bqm = dimod.BinaryQuadraticModel("SPIN")
    h = 3.0
    for i in range(N):
        bqm.add_variable(i, -h)
    return bqm

def interpolate_bqm(initial, problem, s):
    bqm = dimod.BinaryQuadraticModel("SPIN")
    for v in initial.variables:
        lin = (1.0 - s) * initial.linear[v] + s * problem.linear[v]
        bqm.add_variable(v, lin)
    for (u, v), q in initial.quadratic.items():
        q_val = (1.0 - s) * q + s * problem.quadratic.get((u, v), 0.0)
        bqm.add_interaction(u, v, q_val)
    for (u, v), q in problem.quadratic.items():
        if (u, v) not in initial.quadratic:
            bqm.add_interaction(u, v, s * q)
    return bqm

## Adiabatic path

In [ ]:
initial = build_initial_bqm()
problem = build_problem_bqm()
sampler = neal.SimulatedAnnealingSampler()

path = []
for step, s in enumerate(np.linspace(0.0, 1.0, 11)):
    bqm = interpolate_bqm(initial, problem, s)
    response = sampler.sample(bqm, num_reads=50,
                              num_sweeps=max(10, int(500 * s)),
                              initial_states=[{i: 1 for i in range(N)}] * 50)
    best = response.first
    sample = best.sample
    bits = "".join(str(sample.get(i, 0)) for i in range(N))
    energies = [d.energy for d in response.data(["energy"])]
    path.append({"s": s, "bits": bits, "cut": cut_size(bits),
                 "E_min": min(energies), "E_mean": np.mean(energies),
                 "E_max": max(energies)})

print(f"{'s':>5}  {'E_min':>9}  {'E_mean':>9}  {'E_max':>9}  {'best':>6}  {'cut':>3}")
for row in path:
    print(f"{row['s']:5.2f}  {row['E_min']:+9.4f}  {row['E_mean']:+9.4f}  "
          f"{row['E_max']:+9.4f}  {row['bits']:>6}  {row['cut']:3d}")

## Final result

In [ ]:
final = path[-1]
print(f"Best sample: {final['bits']}")
print(f"Best energy: {final['E_min']:+.4f}")
print(f"Cut size:    {final['cut']} / 4")